In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [ ]:
import torch

from tokenizers import Tokenizer

from storyforge.config import load_config
from storyforge.model import ForgeGPT
from storyforge.generation import generate
import copy
import hashlib
from pathlib import Path
import json

In [3]:
config = load_config("configs/default.toml")

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [6]:
tokenizer = Tokenizer.from_file(
    config["tokenizer"]["path"]
)

In [7]:
torch.manual_seed(config["seed"])

In [ ]:
model = ForgeGPT(
    config,
    vocab_size=tokenizer.get_vocab_size(),
).to(device)

In [9]:
for index, prompt in enumerate(config["generation"]["prompts"]):
    text = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        config=config,
        device=device,
        seed=config["generation"]["seed"] + index,
    )

    print(f"\nPROMPT {index + 1}: {prompt}")
    print(text)


PROMPT 1: One night, a little cat named Luna saw a spaceship in the garden.
One night, a little cat named Luna saw a spaceship in the garden. left slips gratitude compassionate spentured ventured streng themselves I card eaten careless eng Jayitsout careless careless balloons seek Maybe Lee forests enjoyingvet Be become size intelligent enjoyiday unicornz sungsit pictures Ray Ray mine instrum helmets stool trop blood marked adding vetupRosie fat lid attention�ereIs� horses disturb groceryact grapesomes let redissasure intelligent challen tastes thumbromise hesit perm Josh Then Then steal size chirp street boom gesture gesture gesture season wireeke gasped eaten Teddy mind mindeez fasteroop drawerling End wifeeez died chirp comet instructions nursesately chirp while no stom coach Bir suitcase figures porch owned bookcase Jones paintbrush microscopeorter it craft craft Br gratitude�Is ga No herbs cows imag below eng looks try happ finishing raz familiar shined surprise gladYeah hallway 

In [11]:
config = load_config("configs/default.toml")

In [12]:
tokenizer_path = Path(config["tokenizer"]["path"])

In [13]:
tokenizer_hash = hashlib.sha256(
    tokenizer_path.read_bytes()
).hexdigest()

In [14]:
checkpoint_path = Path(
    config["checkpoints"]["initial_path"]
)

In [15]:
checkpoint_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

In [16]:
checkpoint = {
    "model": model.state_dict(),
    "config": copy.deepcopy(config),
    "vocab_size": tokenizer.get_vocab_size(),
    "tokenizer_sha256": tokenizer_hash,
    "processed_tokens": 0,
}

In [17]:
torch.save(checkpoint, checkpoint_path)

In [18]:
print("Saved initial checkpoint:", checkpoint_path)

Saved initial checkpoint: runs/baseline/initial.pt


In [20]:
initial_samples = []

for index, prompt in enumerate(config["generation"]["prompts"]):
    seed = config["generation"]["seed"] + index

    text = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        config=config,
        device=device,
        seed=seed,
    )

    initial_samples.append({
        "prompt": prompt,
        "seed": seed,
        "text": text,
    })

In [21]:
sample_history = {
    "initial": {
        "processed_tokens": 0,
        "tokenizer_sha256": tokenizer_hash,
        "generation_config": copy.deepcopy(config["generation"]),
        "samples": initial_samples,
    },
}

In [22]:
samples_path = Path(
    config["generation"]["samples_path"]
)

In [23]:
samples_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

In [24]:
with samples_path.open("w", encoding="utf-8") as file:
    json.dump(
        sample_history,
        file,
        indent=2,
        ensure_ascii=False,
    )

In [25]:
print("Saved sample history:", samples_path)
print("Number of prompts:", len(initial_samples))

Saved sample history: runs/baseline/samples.json
Number of prompts: 3
